# 25 — T3: kiểm chứng trạng thái đầu và mixer domain-wall

Giữ **đúng a3_dev_n5, 5 task × 3 host, 25 qubit**, cùng QUBO/penalty/y/slack của Notebook24.

Notebook này xác nhận tính đúng và đo chi phí trước thí nghiệm T4. Ba cấu hình: chuẩn + X; domain hợp lệ + X; domain hợp lệ + mixer AB bảo toàn miền. **Không chạy optimizer, không nhập góc đã học, không kết luận phương án thắng từ các góc cố định.** Đọc N25_README_FIRST.md trước.

## 1. Môi trường và vị trí repo

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import sys, json, hashlib, zipfile
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'src/domain_wall_probe.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Chép đủ gói Notebook25 vào thư mục gốc repo đã có Notebook24.')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.domain_wall_mixer import validate_fixed_instance, VARIANTS
from src.domain_wall_probe import run_probe, certify_pair
print('Python:', sys.executable)
print('Repo:', ROOT)

Python: c:\Users\Khiem\anaconda3\envs\nt409-khiem\python.exe
Repo: c:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement


## 2. Input cố định — không thay CPU/RAM, task/host hoặc thứ tự

In [2]:
instance = json.loads((ROOT/'data/benchmarks/nt409_shot_noise_v2/instance.json').read_text(encoding='utf-8'))
validate_fixed_instance(instance)
display(pd.DataFrame(instance['tasks']))
display(pd.DataFrame(instance['hosts']))
print('5 task × 3 host; 25 qubit = 10 assignment + 3 y + 12 slack')

,id,cpu,ram
0,t0,1,1
1,t1,1,1
2,t2,1,1
3,t3,1,1
4,t4,1,2


,id,cpu,ram
0,h0,2,2
1,h1,3,3
2,h2,3,3


5 task × 3 host; 25 qubit = 10 assignment + 3 y + 12 slack


## 3. Kiểm tra khối mixer

Mỗi task có ba mã (u,v) hợp lệ (0,0), (1,0), (1,1). A trộn (0,0)↔(1,0), rồi B trộn (1,0)↔(1,1). Mỗi khối là unitary và giữ miền. U=B(beta)A(beta); không gọi tích này là exp[-i beta(A+B)]. Các khối 4×4 dùng để chứng minh cổng, không phải đổi bài toán placement thành ít task hơn.

Trạng thái đầu domain là tích của năm chồng chập đều trên ba mã; 15 qubit y/slack vẫn bắt đầu ở |+>. Hợp lệ miền mã hóa chưa đồng nghĩa thỏa CPU/RAM hoặc slack.

In [3]:
certificate = certify_pair()
display(certificate)
assert certificate['connected']
assert certificate['max_leakage'] <= 1e-12

{'max_matrix_error': 4.440892098500626e-16,
 'max_leakage': 0.0,
 'connected': True,
 'circuit_order': 'A then B; U=B(beta)A(beta)',
 'warning': 'Product of exponentials, not a single exponential of A+B.'}

## 4. Chạy phép kiểm chứng đầy đủ 25 qubit

Chạy tuần tự p=1,2 × ba cấu hình. Có ba kiểm tra trạng thái đầu ở góc0 và sáu lần đánh giá ở góc cố định chưa học. Không có lượt tối ưu; không có 600 evaluation huấn luyện. Giữ máy không sleep. Guard RAM kiểm tra giữa các lời gọi native, không phải cơ chế ngắt tức thì. Mỗi lần Run All tạo thư mục mới; kết quả lần trước được giữ.

In [4]:
THREADS = 4
MIN_AVAILABLE_GIB = 4.0
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT_DIR = ROOT/'data/reference_results/phase5_domainwall_t3'/('probe_'+stamp)
print('Output:', OUT_DIR)
result = run_probe(ROOT, OUT_DIR, threads=THREADS, min_available_gib=MIN_AVAILABLE_GIB)

Output: c:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_t3\probe_20261025T033012396324Z
T3 p=1 standard_x: build frozen 25-qubit circuit
  PASS: P_domain=0.194237568369; one eval=3.940s
T3 p=1 valid_x: build frozen 25-qubit circuit
  PASS: P_domain=0.611108840818; one eval=3.510s
T3 p=1 valid_dw: build frozen 25-qubit circuit
  PASS: P_domain=1.000000000000; one eval=3.717s
T3 p=2 standard_x: build frozen 25-qubit circuit
  PASS: P_domain=0.196730417442; one eval=6.975s
T3 p=2 valid_x: build frozen 25-qubit circuit
  PASS: P_domain=0.376628153778; one eval=6.370s
T3 p=2 valid_dw: build frozen 25-qubit circuit
  PASS: P_domain=1.000000000000; one eval=6.649s


## 5. Đọc kết quả — kiểm tra leakage và chi phí, không xếp hạng chất lượng

In [5]:
assert result['status'] == 'PASS' and len(result['records']) == 6
print('Training evaluations:', result['training_evaluations'])
display(pd.DataFrame(result['initial_checks']))
table = pd.DataFrame(result['records'])
display(table[['variant','p','p_domain','p_valid','p_opt','mean_energy',
               'evaluation_seconds','setup_seconds','compiled_depth','rss_after_evaluation_mib']])
print('valid_dw phải có P_domain gần1 ở cả p=1 và2. Các P_opt ở góc chưa học không chọn phương án thắng.')
print('RSS là số đo sau evaluation, không phải đỉnh RAM; thiếu số đo được lưu null.')

Training evaluations: 0


,variant,p_domain,p_valid,p_opt,p_ground,mean_energy,probability_sum
0,standard_x,0.237305,0.011963,0.000977,2.384186e-07,248.5,1.0
1,valid_x,1.000000,0.050412,0.004115,1.004694e-06,185.5,1.0
2,valid_dw,1.000000,0.050412,0.004115,1.004694e-06,185.5,1.0


,variant,p,p_domain,p_valid,p_opt,mean_energy,evaluation_seconds,setup_seconds,compiled_depth,rss_after_evaluation_mib
0,standard_x,1,0.194238,0.000870,0.000044,548.977473,3.939952,44.779977,37,560.933594
1,valid_x,1,0.611109,0.003445,0.000262,351.375418,3.509516,41.480928,38,566.941406
2,valid_dw,1,1.000000,0.005988,0.000330,334.250037,3.716934,41.478587,39,568.011719
3,standard_x,2,0.196730,0.000249,0.000066,637.830905,6.974786,42.009417,63,567.347656
4,valid_x,2,0.376628,0.000667,0.000210,433.873143,6.369783,46.005578,64,568.765625
5,valid_dw,2,1.000000,0.002348,0.000343,395.007873,6.648710,44.956403,69,571.156250


valid_dw phải có P_domain gần1 ở cả p=1 và2. Các P_opt ở góc chưa học không chọn phương án thắng.
RSS là số đo sau evaluation, không phải đỉnh RAM; thiếu số đo được lưu null.


## 6. Xuất kết quả để gửi kiểm tra

In [6]:
checks = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
          for p in OUT_DIR.iterdir() if p.is_file() and p.name != 'SHA256.json'}
(OUT_DIR/'SHA256.json').write_text(json.dumps(checks, indent=2)+'\n', encoding='utf-8')
archive = OUT_DIR.with_suffix('.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT_DIR.iterdir()):
        if p.is_file(): z.write(p, arcname=p.name)
with zipfile.ZipFile(archive) as z: assert z.testzip() is None
print('Gửi notebook đã Save với output và ZIP:', archive)

Gửi notebook đã Save với output và ZIP: c:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_t3\probe_20261025T033012396324Z.zip
